# Assignment 2: Basic Data Pre-Processing (UCI Dataset)

**Objective:** Perform core data-preprocessing operations on a real dataset using Python.

**Dataset Used:** Wine Quality Dataset — UCI Machine Learning Repository  
**Source:** https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv  
**Description:** This dataset contains physicochemical properties (e.g., acidity, pH, alcohol) of 1599 red wine samples, along with a quality score (0–10) assigned by human tasters.

---

## 🔹 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully.')

## 🔹 Step 2: Load Dataset from UCI Repository

In [ ]:
# Wine Quality Dataset (Red Wine) from UCI ML Repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
df = pd.read_csv(url, sep=';')
print('Dataset loaded successfully!')
df.head()

## 🔹 Step 3: Dataset Overview

In [ ]:
print('Shape (rows, columns):', df.shape)
print('\nColumn Names:', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

> **Note:** The Wine Quality dataset has 1599 rows and 12 columns. All features are numeric (physicochemical tests), and the target column is `quality` (integer score from 3 to 8).

## 🔹 Step 4: Data Types & Summary Statistics

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary
df.describe()

> **Observation:** All columns are of numeric type (float64 / int64). Notice that `total sulfur dioxide` and `free sulfur dioxide` have large ranges — this suggests the need for feature scaling.

## 🔹 Step 5: Check for Missing Values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

## 🔹 Step 6: Artificially Introduce & Handle Missing Values

Since the Wine Quality dataset has no missing values, we will artificially introduce some to demonstrate handling strategies.

In [ ]:
# Artificially introduce missing values (for demonstration)
np.random.seed(42)
df_with_nulls = df.copy()

# Introduce ~5% missing values in 3 columns
for col in ['pH', 'alcohol', 'sulphates']:
    null_indices = np.random.choice(df_with_nulls.index, size=int(0.05 * len(df_with_nulls)), replace=False)
    df_with_nulls.loc[null_indices, col] = np.nan

print('Missing values after introduction:')
print(df_with_nulls.isnull().sum())

In [ ]:
# Handle missing values: fill numeric columns with mean
for col in df_with_nulls.select_dtypes(include=['float64', 'int64']).columns:
    df_with_nulls[col].fillna(df_with_nulls[col].mean(), inplace=True)

# Handle missing values: fill categorical columns with mode
for col in df_with_nulls.select_dtypes(include=['object']).columns:
    df_with_nulls[col].fillna(df_with_nulls[col].mode()[0], inplace=True)

df = df_with_nulls.copy()
print('Missing values after handling:')
print(df.isnull().sum())

> **Strategy:** Numeric columns are filled with their **mean**, which is robust against skewed distributions. Categorical columns are filled with the **mode** (most frequent value).

## 🔹 Step 7: Remove Duplicate Records

In [ ]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]
print(f'Rows before: {before}')
print(f'Rows after:  {after}')
print(f'Duplicates removed: {before - after}')

## 🔹 Step 8: Create a Categorical Column & Encode It

The Wine Quality dataset has no categorical columns. We will derive a meaningful one from the `quality` score.

In [ ]:
# Derive a categorical 'quality_label' column
def label_quality(score):
    if score <= 4:
        return 'low'
    elif score <= 6:
        return 'medium'
    else:
        return 'high'

df['quality_label'] = df['quality'].apply(label_quality)
print('Value counts for quality_label:')
print(df['quality_label'].value_counts())
df[['quality', 'quality_label']].head(10)

In [ ]:
# Encode the categorical column using LabelEncoder
encoder = LabelEncoder()
df['quality_label_encoded'] = encoder.fit_transform(df['quality_label'])

print('Encoding mapping:')
for label, code in zip(encoder.classes_, encoder.transform(encoder.classes_)):
    print(f'  {label} → {code}')

df[['quality_label', 'quality_label_encoded']].head(10)

> **Note:** `LabelEncoder` converts string labels to integers. Here: `high=0`, `low=1`, `medium=2` (alphabetical order). For multi-class ML tasks, one-hot encoding (`pd.get_dummies`) is preferred to avoid implying ordinal relationships.

## 🔹 Step 9: Detect & Handle Outliers (IQR Method)

In [ ]:
# Detect outliers using IQR method
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

print('Outlier count per column (IQR method):')
outlier_counts = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_counts[col] = count
    print(f'  {col}: {count} outliers')

In [ ]:
# Cap outliers using IQR clipping (Winsorization)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

print('Outliers capped using IQR Winsorization.')
df.describe()

> **Strategy:** Instead of dropping outlier rows (which loses data), we **clip/cap** them to the IQR boundary. This is called Winsorization and is commonly used in practice.

## 🔹 Step 10: Feature Scaling

In [ ]:
# Scale all numeric feature columns (exclude target and encoded label)
feature_cols = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
    'density', 'pH', 'sulphates', 'alcohol'
]

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print('Feature scaling applied (StandardScaler).')
df[feature_cols].describe().round(4)

> **StandardScaler** transforms each feature to have **mean = 0** and **standard deviation = 1**. This ensures no single feature dominates during model training due to differing units/magnitudes (e.g., `total sulfur dioxide` has much larger values than `density`).

## 🔹 Step 11: Final Cleaned Dataset Overview

In [ ]:
print('Final Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nMissing Values:', df.isnull().sum().sum())
df.head()

## 🔹 Step 12: Save Final Preprocessed Dataset

In [ ]:
df.to_csv('assignment2_cleaned_wine_quality.csv', index=False)
print('Saved: assignment2_cleaned_wine_quality.csv')

---
## ✅ Summary of Preprocessing Steps

| Step | Operation | Details |
|------|-----------|----------|
| 1 | Import Libraries | pandas, numpy, sklearn |
| 2 | Load Dataset | Red Wine Quality – UCI (1599 rows × 12 cols) |
| 3 | Dataset Overview | Shape, column names |
| 4 | Data Types & Stats | df.info(), df.describe() |
| 5 | Check Missing Values | df.isnull().sum() |
| 6 | Handle Missing Values | Mean (numeric), Mode (categorical) |
| 7 | Remove Duplicates | drop_duplicates() |
| 8 | Categorical Encoding | Derived quality_label, LabelEncoder |
| 9 | Outlier Detection & Treatment | IQR method + Winsorization (clipping) |
| 10 | Feature Scaling | StandardScaler (mean=0, std=1) |
| 11 | Final Overview | Shape, nulls, head |
| 12 | Save Dataset | CSV export |

---
### ✔ End of Assignment 2
Make sure to explain outputs briefly while submitting.